Note, to run this notebook you need to slightly change the format for starting a coiled session:
> `AWS_ACCESS_KEY_ID="" AWS_SECRET_ACCESS_KEY="" AWS_SESSION_TOKEN="" AWS_PROFILE="" uv run coiled notebook start --vm-type r8g.8xlarge --sync --sync-ignore .venv`

In [1]:
import icechunk

from srm.config import _icechunk_storage_for_path
from srm.qa_flags import (
    ATTRS_TIME_INVARIANT,
    ATTRS_TIME_VARYING,
    FLAG_LIST_TIME_INVARIANT,
    FLAG_LIST_TIME_VARYING,
    combine_intermediate_flags,
    discover_leaves,
    parse_tag,
    write_final_qa_flags,
)

In [2]:
flag_time_invariant_name = ATTRS_TIME_INVARIANT["short_name"]
flag_time_varying_name = ATTRS_TIME_VARYING["short_name"]

# A. Define what data arrays exist to traverse

In [4]:
# --- Run parameters --------------------------------------------------------
# This regional South-Africa-box run covers three GCMs, each written to its own icechunk store
# (same bucket/branch, named by GCM the same way srm.cache.ArtifactCache names pipeline output
# stores). Looping over GCMS -- rather than hardcoding one, as this notebook used to -- is what
# lets every leaf be compared against ITS OWN GCM's catalog and lineage instead of silently
# reusing whichever GCM happened to be hardcoded.


VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds"]  # , "hurs"]

In [ ]:
GCMS = ["CESM2-WACCM"]

BRANCH = "pr-638-global"
ROOT_DIR = "s3://carbonplan-srm/scratch/output/qa/"
STORE_SUBSET_ID = "global"

# BRANCH = "v0.13.0"
# ROOT_DIR = "s3://us-west-2.opendata.source.coop/carbonplan/srm-downscaling/output/production/"
# STORE_SUBSET_ID = "global"

In [ ]:
# BUCKET = "carbonplan-srm"
# PREFIX = "scratch/output/qa-intermediate-flags/v0.13.0-global"

PLOT_FLAG_MAPS = True
BUCKET = "carbonplan-srm"
PREFIX = "scratch/output/qa-intermediate-flags"

In [9]:
import coiled
from frisky import hijack

cluster = coiled.Cluster(
    name="srm-qaqc-flags-step3",
    region="us-west-2",
    n_workers=12,
    worker_vm_types=["m8gn.xlarge"],
    scheduler_vm_types="c8g.xlarge",
    spot_policy="spot_with_fallback",
    use_best_zone=True,
    tags={"Project": "SRM"},
    worker_options={"nthreads": 8},
    environ={"ZARR_ASYNC__CONCURRENCY": "128"},
)

client = hijack(cluster.get_client())
client

[2026-08-27 15:16:04,775][INFO    ][coiled] Fetching latest package priorities...
[2026-08-27 15:16:04,776][INFO    ][coiled.package_sync] Resolving your local /Users/clairezarakas/Documents/science/srm-downscaling/uv.lock Python environment...
[2026-08-27 15:16:08,868][INFO    ][coiled.package_sync] Scanning 293 python packages...
[2026-08-27 15:16:09,193][INFO    ][coiled] Running pip check...
[2026-08-27 15:16:09,550][INFO    ][coiled] Validating environment...
[2026-08-27 15:16:17,815][INFO    ][coiled] Creating wheel for ~/Documents/science/srm-downscaling/src...
[2026-08-27 15:16:17,914][INFO    ][coiled] Creating wheel for srm...
[2026-08-27 15:16:31,424][INFO    ][coiled] Uploading coiled_local_src...
[2026-08-27 15:16:38,645][INFO    ][coiled] Uploading srm...
[2026-08-27 15:16:51,765][INFO    ][coiled] Creating software environment...
[2026-08-27 15:16:58,403][INFO    ][coiled] Creating Cluster (name: srm-qaqc-flags-nrh, https://cloud.coiled.io/clusters/1973754 ). This usuall

<frisky.Client: scheduler="wss://cluster-nsols.dask.host/dHQauxvzIzX1zo9X/frisky-comm?__frisky_dial_host=35.85.60.70" id="client-0">

In [10]:
[_, tags, _, _, _, tags_np, _, _] = discover_leaves(gcms=GCMS, branch=BRANCH, root_dir=ROOT_DIR, store_subset_id=STORE_SUBSET_ID)

opened 1/1 stores on branch 'pr-638-global': CESM2-WACCM
196 leaves across 1 GCMs


# Write out final overall flags

In [13]:
tag = "CESM2-WACCM_rsds_ssp245_003_qdmsd"

[overall_flag_time_varying, overall_flag_time_invariant] = combine_intermediate_flags(
    tag=tag,
    flag_list_time_varying=FLAG_LIST_TIME_VARYING,
    flag_list_time_invariant=FLAG_LIST_TIME_INVARIANT,
    bucket=BUCKET,
    prefix=PREFIX,
)

lat = overall_flag_time_invariant.lat
lon = overall_flag_time_invariant.lon

In [16]:
print(len(tags))
# This loop takes about 15 minutes to run on v0.13.0 (31 global data arrays)
OVERWRITE = False

repos: dict[str, icechunk.Repository] = {}
for gcm in GCMS:
    repos[gcm] = icechunk.Repository.open(_icechunk_storage_for_path(f"{ROOT_DIR}{gcm}-ERA5-{STORE_SUBSET_ID}.icechunk"))

for i, tag in enumerate(tags):
    print(tag)
    [gcm, var, scenario, ens, method] = parse_tag(tag)
    print("calculating flags")
    [overall_flag_time_varying, overall_flag_time_invariant] = combine_intermediate_flags(
        tag=tag,
        flag_list_time_varying=FLAG_LIST_TIME_VARYING,
        flag_list_time_invariant=FLAG_LIST_TIME_INVARIANT,
        bucket=BUCKET,
        prefix=PREFIX,
    )

    group = f"{method}/{scenario}/{var}/{ens}"
    session = repos[gcm].writable_session(BRANCH)

    print("  writing flag_time_varying")
    write_final_qa_flags(
        session=session,
        group=group,
        flag_data=overall_flag_time_varying,
        flag_name=flag_time_varying_name,
        attrs=ATTRS_TIME_VARYING,
        overwrite=OVERWRITE,
    )

    print("  writing flag_time_invariant")
    write_final_qa_flags(
        session=session,
        group=group,
        flag_data=overall_flag_time_invariant,
        flag_name=flag_time_invariant_name,
        attrs=ATTRS_TIME_INVARIANT,
        overwrite=OVERWRITE,
    )

    commit = session.commit(f"write qa flags for {tag}")
    print(f"    commit {commit}")

71
CESM2-WACCM_pr_ssp245_001_qdmsd
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit 2SG8R1RGNY7DW9F2W7KG
CESM2-WACCM_pr_ssp245_009_qdmsd
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit H0DWPQF1C7VF0C19NVEG
CESM2-WACCM_pr_ssp245_004_qdmsd
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit K7G5F2Z93PZ13X4G5ZHG
CESM2-WACCM_pr_ssp245_007_qdmsd
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit NJKKBF9FQKHJCVR9C6BG
CESM2-WACCM_pr_ssp245_010_qdmsd
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit 6KVAG1B585GCDJRG3110
CESM2-WACCM_pr_ssp245_003_qdmsd
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit CGMC126VKB7AQVJX06M0
CESM2-WACCM_pr_ssp245_005_qdmsd
calculating flags
  writing flag_time_varying
  writing flag_time_invariant
    commit T37VWC0BN53B6G4BMXS0
CESM2-WACCM_pr_ss

In [17]:
if cluster is not None:
    cluster.shutdown()
else:
    print("no cluster was created (cached run); nothing to shut down")

[2026-08-27 16:51:34,351][INFO    ][coiled] Cluster 1973754 deleted successfully.
